In [1]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [3]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Previous campaign
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    df['pdays_recent'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)
    df['multiple_prev_contacts'] = (df['previous'] > 2).astype(int)
    # Call duration
    df['very_short_call'] = (df['duration'] < 30).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    df['medium_call'] = ((df['duration'] >= 60) & (df['duration'] <= 300)).astype(int)
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    # Duration buckets as a single ordinal feature
    df['duration_bucket'] = pd.cut(
        df['duration'],
        bins=[-1, 30, 60, 180, 300, 600, 99999],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)
    # Balance
    df['debt'] = (df['balance'] < 0).astype(int)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['medium_balance'] = ((df['balance'] > 0) & (df['balance'] <= 1000)).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['very_high_balance'] = (df['balance'] > 5000).astype(int)
    df['log_balance'] = np.log1p(df['balance'].clip(lower=0))
    # Campaign pressure
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['log_campaign'] = np.log1p(df['campaign'])
    # Age
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    # Interactions
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    df['success_signal'] = ((df['duration'] > 300) & (df['poutcome'] == 'SUC')).astype(int)
    df['warm_lead'] = ((df['contacted_recently'] == 1) & (df['prev_success'] == 1)).astype(int)
    df['cold_lead'] = ((df['never_contacted'] == 1) & (df['short_call'] == 1)).astype(int)
    # Month/quarter
    df['q1'] = df['month'].isin([1, 2, 3]).astype(int)
    df['q2'] = df['month'].isin([4, 5, 6]).astype(int)
    df['q3'] = df['month'].isin([7, 8, 9]).astype(int)
    df['q4'] = df['month'].isin([10, 11, 12]).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA shape: ', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 49)
TEST_DATA shape:  (19893, 49)


In [4]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']

num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'never_contacted',
            'pdays_recent', 'multiple_prev_contacts',
            'very_short_call', 'short_call', 'medium_call',
            'long_call', 'very_long_call', 'duration_bucket',
            'debt', 'has_balance', 'medium_balance',
            'high_balance', 'very_high_balance', 'log_balance',
            'first_contact', 'over_contacted', 'log_campaign',
            'is_young', 'is_middle_age', 'is_retired_age',
            'long_call_prev_success', 'long_call_never_contacted',
            'high_balance_long_call', 'success_signal',
            'warm_lead', 'cold_lead',
            'q1', 'q2', 'q3', 'q4']

ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    num_df = df[num_cols].copy()
    return pd.concat([cat_enc, num_df], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('Class distribution — 0:', (y_train==0).sum(), '| 1:', (y_train==1).sum())

X_train shape: (29839, 49)
X_test  shape: (19893, 49)
Class distribution — 0: 26353 | 1: 3486


In [5]:
from sklearn.model_selection import StratifiedKFold

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc  = X_te.copy()
    global_mean = y_tr.mean()

    for col in cols:
        oof     = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))

        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = (
                y_tr.iloc[fold_tr_idx]
                .groupby(X_tr[col].iloc[fold_tr_idx])
                .mean()
            )
            oof[fold_val_idx] = (
                X_tr[col].iloc[fold_val_idx]
                .map(means).fillna(global_mean).values
            )
            te_vals += (
                X_te[col].reset_index(drop=True)
                .map(means).fillna(global_mean).values / n_splits
            )

        X_tr_enc[col + '_te'] = oof
        X_te_enc[col  + '_te'] = te_vals

    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)

print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 57)
X_test_te  shape: (19893, 57)


In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()  # ~7.56

# Best params from previous Optuna run
hgbm_params = {
    'learning_rate': 0.016782184919286597,
    'max_iter': 1117,
    'max_leaf_nodes': 26,
    'max_depth': 5,
    'min_samples_leaf': 72,
    'l2_regularization': 2.2767559805786015,
}

xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 5,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 2.0,
    'scale_pos_weight': scale_pos,
    'eval_metric': 'logloss',
    'random_state': 42,
    'n_jobs': -1,
}

lgbm_params = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 5,
    'num_leaves': 26,
    'min_child_samples': 72,
    'reg_lambda': 2.0,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

# OOF arrays
oof_hgbm = np.zeros(len(y_train))
oof_xgb  = np.zeros(len(y_train))
oof_lgbm = np.zeros(len(y_train))

# Test prediction arrays (average across folds)
test_hgbm = np.zeros(len(X_test_te))
test_xgb  = np.zeros(len(X_test_te))
test_lgbm = np.zeros(len(X_test_te))

X_arr = X_train_te.values
X_te_arr = X_test_te.values

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    # HGBM
    m_hgbm = HistGradientBoostingClassifier(
        class_weight='balanced', random_state=42,
        early_stopping=False, **hgbm_params
    )
    m_hgbm.fit(X_tr, y_tr)
    oof_hgbm[val_idx] = m_hgbm.predict_proba(X_val)[:, 1]
    test_hgbm += m_hgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    # XGBoost
    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb += m_xgb.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    # LightGBM
    m_lgbm = LGBMClassifier(**lgbm_params)
    m_lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = m_lgbm.predict_proba(X_val)[:, 1]
    test_lgbm += m_lgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')

print('\nOOF Balanced Accuracy per model:')
for name, oof in [('HGBM', oof_hgbm), ('XGB', oof_xgb), ('LGBM', oof_lgbm)]:
    best_ba, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.9, 0.005):
        ba = balanced_accuracy_score(y_train, (oof >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    print(f'  {name}: {best_ba:.4f}  (best t={best_t:.3f})')

Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

OOF Balanced Accuracy per model:
  HGBM: 0.8736  (best t=0.415)
  XGB: 0.8727  (best t=0.420)
  LGBM: 0.8715  (best t=0.415)


In [7]:
# Try different blend weights on OOF — no leakage since these are OOF
best_ba, best_weights, best_threshold = 0.0, (1, 1, 1), 0.5

for w_h in [1, 2, 3]:
    for w_x in [1, 2, 3]:
        for w_l in [1, 2, 3]:
            total = w_h + w_x + w_l
            blended = (w_h*oof_hgbm + w_x*oof_xgb + w_l*oof_lgbm) / total
            for t in np.arange(0.1, 0.9, 0.005):
                ba = balanced_accuracy_score(y_train, (blended >= t).astype(int))
                if ba > best_ba:
                    best_ba = ba
                    best_weights = (w_h, w_x, w_l)
                    best_threshold = t

print(f'Best OOF Balanced Accuracy: {best_ba:.4f}')
print(f'Best weights (HGBM, XGB, LGBM): {best_weights}')
print(f'Best threshold: {best_threshold:.3f}')

Best OOF Balanced Accuracy: 0.8740
Best weights (HGBM, XGB, LGBM): (1, 3, 1)
Best threshold: 0.455


In [8]:
w_h, w_x, w_l = best_weights
total = w_h + w_x + w_l

test_blend = (w_h*test_hgbm + w_x*test_xgb + w_l*test_lgbm) / total
test_preds = (test_blend >= best_threshold).astype(int)

print(f'Prediction distribution — 0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')
print(f'OOF CV used for threshold: {best_ba:.4f}')

Prediction distribution — 0: 15220, 1: 4673
OOF CV used for threshold: 0.8740


In [10]:
submission = pd.DataFrame({
    'id':           TEST_DATA.index,
    'subscription': test_preds
})
submission.to_csv('submissionx.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             0
